In [1]:
# ============================================================
# FILE 7 — Batch RF Model → CSV Converter
# Automatically processes ALL .joblib models in the input folder
# ============================================================

import os
import re
import time
import glob
import joblib
import datetime
import ee
from geemap import ml



import yaml

# ------------------------------------------------------------
# 1. Load Configuration
# ------------------------------------------------------------
with open('config.yaml', 'r') as file:
    config = yaml.safe_load(file)




PROJECT_ID = config['gee']['project_id']
INPUT_FOLDER = config['paths']['models_joblib_dir']
OUTPUT_FOLDER = config['paths']['models_csv_dir']
NUM_PROCESSES = config['parameters']['num_processes']


# ------------------------------------------------------------
# 1. Initialize Earth Engine
# ------------------------------------------------------------
ee.Authenticate()

try:
    ee.Initialize(
        project=PROJECT_ID,
        opt_url='https://earthengine-highvolume.googleapis.com'
    )
    print('✅ Google Earth Engine initialized successfully!\n')
except ee.EEException as e:
    print('❌ Google Earth Engine failed to initialize!', e)
    raise


os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Expected filename format: rfr_model_YYYY-MM-DD_AEZ_<N>_<VAR>.joblib
FILE_PATTERN = re.compile(r"rfr_model_(.+)_AEZ_(\d+)_(.+)\.joblib$")

# ------------------------------------------------------------
# 3. Discover all .joblib model files
# ------------------------------------------------------------
joblib_files = sorted(glob.glob(os.path.join(INPUT_FOLDER, "*.joblib")))
print(f"Found {len(joblib_files)} model file(s) in: {INPUT_FOLDER}\n")

if not joblib_files:
    raise FileNotFoundError(
        f"No .joblib files found in {INPUT_FOLDER}. "
        "Check your INPUT_FOLDER path."
    )

# ------------------------------------------------------------
# 4. Batch Processing Loop
# ------------------------------------------------------------
processed_count = 0
skipped_count   = 0
failed_count    = 0

for file_path in joblib_files:
    filename = os.path.basename(file_path)

    # --- Parse metadata from filename ---
    match = FILE_PATTERN.match(filename)
    if not match:
        print(f"⚠️  Skipping '{filename}': does not match expected naming convention.")
        failed_count += 1
        continue

    model_date, aez, pred_var = match.groups()
    print(f"{'='*60}")
    print(f"  AEZ: {aez}  |  Nutrient: {pred_var}  |  Date: {model_date}")
    print(f"{'='*60}")

    # --- Load the model ---
    try:
        rf = joblib.load(file_path)
    except Exception as e:
        print(f"❌ Failed to load '{filename}': {e}\n")
        failed_count += 1
        continue

    n_estimators = rf.get_params()['n_estimators']
    max_depth    = rf.get_params()['max_depth']

    # --- Build output CSV path ---
    csv_filename = (
        f"rf_trees_t{n_estimators}_d{max_depth}"
        f"_{model_date}_AEZ_{aez}_{pred_var}.csv"
    )
    output_path = os.path.join(OUTPUT_FOLDER, csv_filename)

    # --- Skip if already converted (crash-resilient) ---
    if os.path.exists(output_path):
        print(f"⏭️  Already exists — skipping: {csv_filename}\n")
        skipped_count += 1
        continue

    # --- Convert RF → strings → CSV ---
    feature_names = rf.feature_names_in_.tolist()
    print(f"   Features  : {feature_names}")
    print(f"   Estimators: {n_estimators}  |  Max depth: {max_depth}")
    print(f"⏳ Converting trees using {NUM_PROCESSES} CPU cores...")

    start_time = time.perf_counter()
    try:
        trees = ml.rf_to_strings(
            rf,
            feature_names,
            processes=NUM_PROCESSES,
            output_mode="REGRESSION"
        )
        ml.trees_to_csv(trees, output_path)

        elapsed = datetime.timedelta(seconds=time.perf_counter() - start_time)
        sizes   = [len(trees[i]) for i in range(len(trees))]
        print(f"   Tree string sizes (sample): {sizes[:5]}{'...' if len(sizes) > 5 else ''}")
        print(f"✅ Saved → {csv_filename}  (Time: {elapsed})\n")
        processed_count += 1

    except Exception as e:
        print(f"❌ Error during conversion of '{filename}': {e}\n")
        failed_count += 1

# ------------------------------------------------------------
# 5. Summary
# ------------------------------------------------------------
print(f"\n{'='*60}")
print(f"  BATCH COMPLETE")
print(f"  ✅ Processed : {processed_count}")
print(f"  ⏭️  Skipped   : {skipped_count}")
print(f"  ❌ Failed    : {failed_count}")
print(f"{'='*60}")

/opt/homebrew/anaconda3/envs/shc-env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


✅ Google Earth Engine initialized successfully!

Found 72 model file(s) in: ./experiments/experiment_5_ndvi_pH_NIRv_LULC/rfr_joblib

  AEZ: 10  |  Nutrient: K  |  Date: 2026-03-11
   Features  : ['temp', 'SI', 'RI', 'precipitation', 'sand05', 'silt05', 'sand515', 'TGSI', 'NDVI_Kharif', 'NDVI_Rabi', 'NDVI_Zaid', 'pH_0-5', 'pH_5-15', 'NIRv_Kharif', 'NIRv_Rabi', 'NIRv_Zaid']
   Estimators: 10  |  Max depth: 16
⏳ Converting trees using 12 CPU cores...


/opt/homebrew/anaconda3/envs/shc-env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/opt/homebrew/anaconda3/envs/shc-env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/opt/homebrew/anaconda3/envs/shc-env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/opt/homebrew/anaconda3/envs/shc-env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/opt/homebrew/anaconda3/envs/shc-env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsD

   Tree string sizes (sample): [341600, 373543, 325805, 326747, 368657]...
✅ Saved → rf_trees_t10_d16_2026-03-11_AEZ_10_K.csv  (Time: 0:00:11.794627)

  AEZ: 10  |  Nutrient: N  |  Date: 2026-03-11
   Features  : ['temp', 'SI', 'RI', 'precipitation', 'sand05', 'silt05', 'sand515', 'TGSI', 'NDVI_Kharif', 'NDVI_Rabi', 'NDVI_Zaid', 'pH_0-5', 'pH_5-15', 'NIRv_Kharif', 'NIRv_Rabi', 'NIRv_Zaid']
   Estimators: 10  |  Max depth: 16
⏳ Converting trees using 12 CPU cores...


/opt/homebrew/anaconda3/envs/shc-env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/opt/homebrew/anaconda3/envs/shc-env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/opt/homebrew/anaconda3/envs/shc-env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/opt/homebrew/anaconda3/envs/shc-env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/opt/homebrew/anaconda3/envs/shc-env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsD

...

   Tree string sizes (sample): [558523, 540414, 491624, 556866, 509537]...
✅ Saved → rf_trees_t10_d16_2026-03-11_AEZ_9_N.csv  (Time: 0:00:41.323331)

  AEZ: 9  |  Nutrient: OC  |  Date: 2026-03-11
   Features  : ['temp', 'SI', 'RI', 'precipitation', 'sand05', 'silt05', 'sand515', 'TGSI', 'NDVI_Kharif', 'NDVI_Rabi', 'NDVI_Zaid', 'pH_0-5', 'pH_5-15', 'NIRv_Kharif', 'NIRv_Rabi', 'NIRv_Zaid']
   Estimators: 10  |  Max depth: 16
⏳ Converting trees using 12 CPU cores...


/opt/homebrew/anaconda3/envs/shc-env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/opt/homebrew/anaconda3/envs/shc-env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/opt/homebrew/anaconda3/envs/shc-env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/opt/homebrew/anaconda3/envs/shc-env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/opt/homebrew/anaconda3/envs/shc-env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsD

   Tree string sizes (sample): [277753, 384111, 376720, 380914, 392330]...
✅ Saved → rf_trees_t10_d16_2026-03-11_AEZ_9_OC.csv  (Time: 0:00:24.265663)

  AEZ: 9  |  Nutrient: P  |  Date: 2026-03-11
   Features  : ['temp', 'SI', 'RI', 'precipitation', 'sand05', 'silt05', 'sand515', 'TGSI', 'NDVI_Kharif', 'NDVI_Rabi', 'NDVI_Zaid', 'pH_0-5', 'pH_5-15', 'NIRv_Kharif', 'NIRv_Rabi', 'NIRv_Zaid']
   Estimators: 10  |  Max depth: 16
⏳ Converting trees using 12 CPU cores...


/opt/homebrew/anaconda3/envs/shc-env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/opt/homebrew/anaconda3/envs/shc-env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/opt/homebrew/anaconda3/envs/shc-env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/opt/homebrew/anaconda3/envs/shc-env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/opt/homebrew/anaconda3/envs/shc-env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsD

   Tree string sizes (sample): [502797, 479067, 533449, 506393, 458885]...
✅ Saved → rf_trees_t10_d16_2026-03-11_AEZ_9_P.csv  (Time: 0:00:38.496284)


  BATCH COMPLETE
  ✅ Processed : 70
  ⏭️  Skipped   : 2
  ❌ Failed    : 0
